# Studi Kasus Webinar ke-5: Human-Computer Interaction & Data Visualization
## Dashboard Analisis Akademik Mahasiswa

Notebook ini mengimplementasikan studi kasus dashboard akademik menggunakan **Python, Pandas, NumPy, Plotly, dan ipywidgets**.

### Tujuan
- Menganalisis data akademik mahasiswa sintetis.
- Menampilkan ringkasan statistik dan visualisasi yang sesuai dengan tujuan analisis.
- Menerapkan prinsip HCI seperti *overview first*, *recognition over recall*, konsistensi, umpan balik, dan filter interaktif.
- Melakukan pengujian sederhana terhadap kebutuhan pengguna.

> **Catatan:** Data yang dibuat di notebook ini bersifat sintetis untuk demonstrasi, bukan data mahasiswa nyata.


## 1. Instalasi pustaka
Jalankan sel berikut jika pustaka yang diperlukan belum tersedia di lingkungan notebook.

In [ ]:
%pip install -q pandas numpy plotly ipywidgets

## 2. Import pustaka

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display, Markdown

# Pengaturan tampilan Pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Pustaka berhasil diimpor.")

## 3. Membuat dataset demonstrasi
Dataset mencakup NIM demo, nama mahasiswa, program studi, semester, IPK, dan persentase kehadiran.

In [ ]:
rng = np.random.default_rng(42)
jumlah = 240

prodi_list = [
    "Informatika",
    "Sistem Informasi",
    "Teknik Sipil",
    "Manajemen"
]
semester_list = [2, 4, 6, 8]

df = pd.DataFrame({
    "NIM": [f"DEMO{i:04d}" for i in range(1, jumlah + 1)],
    "Nama": [f"Mahasiswa {i:03d}" for i in range(1, jumlah + 1)],
    "Program Studi": rng.choice(prodi_list, jumlah),
    "Semester": rng.choice(semester_list, jumlah),
    "IPK": np.clip(rng.normal(3.15, 0.40, jumlah), 1.50, 4.00).round(2),
    "Kehadiran (%)": np.clip(rng.normal(85, 10, jumlah), 40, 100).round(1)
})

df["Kategori IPK"] = pd.cut(
    df["IPK"],
    bins=[0, 2.50, 3.00, 3.50, 4.01],
    labels=["Perlu Perhatian", "Cukup", "Baik", "Sangat Baik"],
    right=False
)

display(df.head(10))
print(f"Jumlah baris: {len(df)}")
print(f"Jumlah kolom: {len(df.columns)}")

## 4. Memahami dan memeriksa data
Pemeriksaan awal membantu memastikan data siap dianalisis.

In [ ]:
print("Informasi dataset:")
df.info()

print("\nJumlah nilai kosong per kolom:")
display(df.isna().sum().to_frame("Nilai kosong"))

print("\nStatistik deskriptif:")
display(df[["IPK", "Kehadiran (%)"]].describe().round(2))

## 5. Ringkasan statistik (*overview first*)
Bagian ini menyajikan informasi utama terlebih dahulu sebelum pengguna mengeksplorasi grafik dan detail.

In [ ]:
ringkasan = pd.DataFrame({
    "Metrik": [
        "Jumlah mahasiswa",
        "Rata-rata IPK",
        "Rata-rata kehadiran",
        "IPK tertinggi",
        "IPK terendah"
    ],
    "Nilai": [
        len(df),
        round(df["IPK"].mean(), 2),
        f"{df['Kehadiran (%)'].mean():.1f}%",
        round(df["IPK"].max(), 2),
        round(df["IPK"].min(), 2)
    ]
})

display(ringkasan)

## 6. Visualisasi 1 — Perbandingan rata-rata IPK
**Bar chart** digunakan untuk membandingkan nilai antarprogram studi.

In [ ]:
ipk_prodi = (
    df.groupby("Program Studi", as_index=False)["IPK"]
      .mean()
      .sort_values("IPK", ascending=False)
)

fig_bar = px.bar(
    ipk_prodi,
    x="Program Studi",
    y="IPK",
    text="IPK",
    title="Perbandingan Rata-rata IPK per Program Studi",
    labels={"Program Studi": "Program Studi", "IPK": "Rata-rata IPK"}
)
fig_bar.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig_bar.update_layout(yaxis_range=[0, 4.3], showlegend=False)
fig_bar.show()

display(ipk_prodi.round(2))

## 7. Visualisasi 2 — Distribusi kategori IPK
**Donut chart** memperlihatkan proporsi setiap kategori terhadap keseluruhan mahasiswa.

In [ ]:
urutan_kategori = ["Perlu Perhatian", "Cukup", "Baik", "Sangat Baik"]
distribusi = (
    df["Kategori IPK"]
    .value_counts()
    .reindex(urutan_kategori, fill_value=0)
    .rename_axis("Kategori IPK")
    .reset_index(name="Jumlah")
)

fig_donut = px.pie(
    distribusi,
    names="Kategori IPK",
    values="Jumlah",
    hole=0.45,
    title="Distribusi Kategori IPK"
)
fig_donut.update_traces(textposition="inside", textinfo="percent+label")
fig_donut.show()

display(distribusi)

## 8. Visualisasi 3 — Hubungan IPK dan kehadiran
**Scatter plot** membantu mengeksplorasi hubungan dua variabel numerik. Hubungan yang terlihat tidak otomatis membuktikan sebab-akibat.

In [ ]:
fig_scatter = px.scatter(
    df,
    x="Kehadiran (%)",
    y="IPK",
    color="Program Studi",
    hover_data=["Nama", "Semester", "Kategori IPK"],
    title="Hubungan IPK dan Tingkat Kehadiran",
    labels={
        "Kehadiran (%)": "Kehadiran (%)",
        "IPK": "Indeks Prestasi Kumulatif"
    }
)
fig_scatter.update_layout(xaxis_range=[0, 105], yaxis_range=[1, 4.2])
fig_scatter.show()

## 9. Visualisasi 4 — Rata-rata IPK berdasarkan semester
**Line chart** menampilkan pola nilai rata-rata pada semester yang tersedia. Karena ini data sintetis, pola tidak mewakili tren nyata.

In [ ]:
ipk_semester = (
    df.groupby("Semester", as_index=False)["IPK"]
      .mean()
      .sort_values("Semester")
)

fig_line = px.line(
    ipk_semester,
    x="Semester",
    y="IPK",
    markers=True,
    title="Rata-rata IPK berdasarkan Semester",
    labels={"Semester": "Semester", "IPK": "Rata-rata IPK"}
)
fig_line.update_layout(
    xaxis=dict(tickmode="array", tickvals=ipk_semester["Semester"].tolist()),
    yaxis_range=[0, 4.3]
)
fig_line.show()

display(ipk_semester.round(2))

## 10. Filter interaktif (*zoom and filter*)
Sel ini menggunakan `ipywidgets`. Jika widget tidak muncul di lingkungan yang digunakan, bagian analisis noninteraktif di atas tetap dapat dijalankan.
Pilih program studi, semester, dan rentang IPK untuk menyaring data.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

prodi_widget = widgets.SelectMultiple(
    options=sorted(df["Program Studi"].unique()),
    value=tuple(sorted(df["Program Studi"].unique())),
    description="Prodi:",
    layout=widgets.Layout(width="350px", height="110px")
)

semester_widget = widgets.SelectMultiple(
    options=sorted(df["Semester"].unique()),
    value=tuple(sorted(df["Semester"].unique())),
    description="Semester:",
    layout=widgets.Layout(width="350px", height="110px")
)

ipk_widget = widgets.FloatRangeSlider(
    value=(1.5, 4.0),
    min=1.5,
    max=4.0,
    step=0.05,
    description="Rentang IPK:",
    readout_format=".2f",
    layout=widgets.Layout(width="500px")
)

output_filter = widgets.Output()

def update_dashboard(change=None):
    with output_filter:
        clear_output(wait=True)
        filtered = df[
            df["Program Studi"].isin(prodi_widget.value)
            & df["Semester"].isin(semester_widget.value)
            & df["IPK"].between(ipk_widget.value[0], ipk_widget.value[1])
        ].copy()

        display(Markdown("### Ringkasan hasil filter"))
        if filtered.empty:
            print("Tidak ada data yang sesuai. Ubah pilihan filter.")
            return

        metrics = pd.DataFrame({
            "Metrik": ["Jumlah mahasiswa", "Rata-rata IPK", "Rata-rata kehadiran"],
            "Nilai": [
                len(filtered),
                round(filtered["IPK"].mean(), 2),
                f"{filtered['Kehadiran (%)'].mean():.1f}%"
            ]
        })
        display(metrics)

        by_prodi = (
            filtered.groupby("Program Studi", as_index=False)["IPK"]
            .mean()
            .sort_values("IPK", ascending=False)
        )
        fig = px.bar(
            by_prodi, x="Program Studi", y="IPK", text="IPK",
            title="Rata-rata IPK setelah filter",
            labels={"IPK": "Rata-rata IPK"}
        )
        fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
        fig.update_layout(yaxis_range=[0, 4.3], showlegend=False)
        fig.show()

        display(Markdown("### Detail mahasiswa"))
        display(
            filtered.sort_values("IPK", ascending=False)
            .reset_index(drop=True)
        )

for widget in [prodi_widget, semester_widget, ipk_widget]:
    widget.observe(update_dashboard, names="value")

display(widgets.VBox([
    prodi_widget,
    semester_widget,
    ipk_widget,
    output_filter
]))
update_dashboard()

## 11. Pencarian detail mahasiswa
Fitur pencarian membantu pengguna menemukan informasi tanpa harus mengingat atau menelusuri seluruh tabel.

In [ ]:
kata_kunci = input("Masukkan NIM atau nama mahasiswa (kosongkan untuk semua): ").strip()

if kata_kunci:
    hasil_cari = df[
        df["NIM"].str.contains(kata_kunci, case=False, na=False)
        | df["Nama"].str.contains(kata_kunci, case=False, na=False)
    ]
else:
    hasil_cari = df.copy()

print(f"Jumlah hasil: {len(hasil_cari)}")
display(hasil_cari.sort_values("IPK", ascending=False).head(30))

## 12. Ekspor data hasil analisis
Simpan dataset atau hasil filter ke CSV untuk analisis lanjutan. File dapat diunduh dari lingkungan notebook setelah sel dijalankan.

In [ ]:
# Ekspor seluruh data demonstrasi
df.to_csv("data_akademik_demo.csv", index=False, encoding="utf-8-sig")

# Contoh ekspor hasil filter berdasarkan pilihan pada bagian interaktif
filtered_export = df[
    df["Program Studi"].isin(prodi_widget.value)
    & df["Semester"].isin(semester_widget.value)
    & df["IPK"].between(ipk_widget.value[0], ipk_widget.value[1])
]
filtered_export.to_csv(
    "data_akademik_hasil_filter.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File berhasil dibuat:")
print("- data_akademik_demo.csv")
print("- data_akademik_hasil_filter.csv")

## 13. Checklist pengujian usability

Gunakan checklist ini ketika meminta pengguna mencoba notebook atau dashboard. Isi hasil pengujian secara nyata; jangan menandai berhasil sebelum diuji.

| No. | Skenario pengujian | Kriteria keberhasilan |
|---|---|---|
| 1 | Menemukan jumlah mahasiswa dan rata-rata IPK | Informasi utama mudah ditemukan |
| 2 | Memilih satu program studi | Ringkasan dan grafik berubah sesuai filter |
| 3 | Mengubah rentang IPK | Data di luar rentang tidak ditampilkan |
| 4 | Mencari nama/NIM | Hasil pencarian sesuai kata kunci |
| 5 | Mengekspor CSV | File dapat dibuka dan isinya sesuai hasil analisis |

### Prinsip HCI yang diterapkan
- **Overview first:** ringkasan statistik ditampilkan sebelum detail.
- **Zoom and filter:** pengguna mempersempit data melalui filter.
- **Details on demand:** tabel detail dapat ditampilkan setelah pengguna menentukan fokus.
- **Recognition over recall:** label dan judul grafik menjelaskan informasi yang ditampilkan.
- **Feedback:** hasil berubah setelah filter atau pencarian diperbarui.
- **Konsistensi:** format angka, judul, dan label digunakan secara seragam.

## 14. Kesimpulan

Notebook ini menunjukkan penerapan prinsip HCI dan visualisasi data melalui dashboard analisis akademik. Bar chart digunakan untuk perbandingan, donut chart untuk proporsi, scatter plot untuk eksplorasi hubungan dua variabel, dan line chart untuk pola berdasarkan semester. Filter dan pencarian membantu pengguna mengeksplorasi data sesuai kebutuhan.

Data bersifat sintetis, sehingga hasilnya hanya untuk demonstrasi. Sebelum digunakan untuk keputusan akademik nyata, sistem perlu diuji bersama pengguna dan menggunakan data yang valid serta dilindungi sesuai kebijakan privasi.
